该文件用于检查文本转换结果。

In [1]:
import json
import pandas as pd
import os
from pathlib import Path

rules_file = "Feature2Txt/output/conversion_rules.json"

if os.path.exists(rules_file):
    print(f"Found conversion rules file: {rules_file}")
    
    with open(rules_file, 'r', encoding='utf-8') as f:
        conversion_rules = json.load(f)
    
    print(f"Total variables in conversion rules: {len(conversion_rules)}")
else:
    print(f"Conversion rules file not found: {rules_file}")
    print("Available files in Feature2Txt/output/:")
    output_dir = Path("Feature2Txt/output")
    if output_dir.exists():
        for file in output_dir.glob("*"):
            print(f"  - {file.name}")
    else:
        print("  Output directory doesn't exist")

Conversion rules file not found: Feature2Txt/output/conversion_rules.json
Available files in Feature2Txt/output/:
  Output directory doesn't exist


In [2]:
import os
print("Current working directory:", os.getcwd())
print("\nDirectory structure:")

feature2txt_dir = Path("Feature2Txt")
if feature2txt_dir.exists():
    print(f"\nFeature2Txt directory contents:")
    for item in sorted(feature2txt_dir.iterdir()):
        if item.is_file():
            print(f"  📄 {item.name}")
        else:
            print(f"  📁 {item.name}/")
            if item.name == "output":
                for subitem in sorted(item.iterdir()):
                    print(f"      {'📄' if subitem.is_file() else '📁'} {subitem.name}")

print(f"\nSearching for conversion_rules files:")
for root, dirs, files in os.walk("."):
    for file in files:
        if "conversion_rules" in file.lower():
            print(f"  Found: {os.path.join(root, file)}")

Current working directory: /home/sxy/adRAG/Feature2Txt

Directory structure:

Searching for conversion_rules files:
  Found: ./output/improved_conversion_rules.json
  Found: ./output/conversion_rules.json


In [3]:
import re

def analyze_conversion_rules(file_path):
    print(f"\n{'='*60}")
    print(f"Analyzing: {file_path}")
    print(f"{'='*60}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        rules = json.load(f)
    
    total_variables = len(rules)
    failed_variables = []
    success_variables = []
    
    print(f"Total variables: {total_variables}")
    
    for var_name, rule in rules.items():
        value_mappings = rule.get("value_mappings", {})
        english_name = rule.get("english_name", "")
        template = rule.get("template", "")
        
        is_failed = False
        
        if len(value_mappings) == 1 and "default" in value_mappings:
            default_value = value_mappings["default"].lower()
            if "unknown" in default_value:
                is_failed = True
        
        expected_simple_name = var_name.replace('_', ' ').title()
        if english_name == expected_simple_name and template == "The patient has {VALUE}":
            is_failed = True
        
        if is_failed:
            failed_variables.append({
                'var_name': var_name,
                'english_name': english_name,
                'value_mappings': value_mappings,
                'template': template
            })
        else:
            success_variables.append({
                'var_name': var_name,
                'english_name': english_name,
                'mappings_count': len(value_mappings),
                'template': template
            })
    
    print(f"Failed conversions: {len(failed_variables)}")
    print(f"Successful conversions: {len(success_variables)}")
    print(f"Success rate: {len(success_variables)/total_variables*100:.1f}%")
    
    return failed_variables, success_variables

files_to_analyze = [
    "./output/conversion_rules.json",
    "./output/improved_conversion_rules.json"
]

all_results = {}
for file_path in files_to_analyze:
    if os.path.exists(file_path):
        failed, success = analyze_conversion_rules(file_path)
        all_results[file_path] = {
            'failed': failed,
            'success': success
        }


Analyzing: ./output/conversion_rules.json
Total variables: 523
Failed conversions: 3
Successful conversions: 520
Success rate: 99.4%

Analyzing: ./output/improved_conversion_rules.json
Total variables: 10
Failed conversions: 0
Successful conversions: 10
Success rate: 100.0%


In [4]:
print("\n" + "="*80)
print("DETAILED ANALYSIS OF FAILED CONVERSIONS")
print("="*80)

for file_path, results in all_results.items():
    failed_vars = results['failed']
    if failed_vars:
        print(f"\n📊 File: {file_path}")
        print(f"Failed variables ({len(failed_vars)}):")
        print("-" * 50)
        
        for i, var in enumerate(failed_vars, 1):
            print(f"\n{i}. Variable: {var['var_name']}")
            print(f"   English Name: {var['english_name']}")
            print(f"   Template: {var['template']}")
            print(f"   Value Mappings: {var['value_mappings']}")
    else:
        print(f"\n✅ File: {file_path}")
        print("   No failed conversions found!")

print(f"\n" + "="*80)
print("EXAMPLES OF SUCCESSFUL CONVERSIONS")
print("="*80)

success_vars = all_results["./output/conversion_rules.json"]['success']
print(f"\nShowing 10 examples from {len(success_vars)} successful conversions:")
print("-" * 60)

for i, var in enumerate(success_vars[:10], 1):
    print(f"\n{i}. Variable: {var['var_name']}")
    print(f"   English Name: {var['english_name']}")
    print(f"   Template: {var['template']}")
    print(f"   Mappings Count: {var['mappings_count']}")

mapping_counts = [var['mappings_count'] for var in success_vars]
print(f"\n📈 MAPPING DISTRIBUTION:")
print(f"   Average mappings per variable: {sum(mapping_counts)/len(mapping_counts):.1f}")
print(f"   Min mappings: {min(mapping_counts)}")
print(f"   Max mappings: {max(mapping_counts)}")

from collections import Counter
mapping_distribution = Counter(mapping_counts)
print(f"\n   Distribution by mapping count:")
for count, freq in sorted(mapping_distribution.items()):
    print(f"     {count} mappings: {freq} variables ({freq/len(success_vars)*100:.1f}%)")


DETAILED ANALYSIS OF FAILED CONVERSIONS

📊 File: ./output/conversion_rules.json
Failed variables (3):
--------------------------------------------------

1. Variable: VISITDAY
   English Name: Visitday
   Template: The patient has {VALUE}
   Value Mappings: {'default': 'unknown visitday'}

2. Variable: OTRLBLI
   English Name: Otrlbli
   Template: The patient has {VALUE}
   Value Mappings: {'default': 'unknown otrlbli'}

3. Variable: REYTCOR
   English Name: Reytcor
   Template: The patient has {VALUE}
   Value Mappings: {'default': 'unknown reytcor'}

✅ File: ./output/improved_conversion_rules.json
   No failed conversions found!

EXAMPLES OF SUCCESSFUL CONVERSIONS

Showing 10 examples from 520 successful conversions:
------------------------------------------------------------

1. Variable: BIRTHYR
   English Name: Year of birth
   Template: The patient {VALUE}
   Mappings Count: 1

2. Variable: RACEX
   English Name: Other Race Specification
   Template: The patient {VALUE}
   Mapp

In [5]:
print("📋 CONVERSION RULES ANALYSIS SUMMARY")
print("=" * 50)

for file_path, results in all_results.items():
    file_name = os.path.basename(file_path)
    failed_count = len(results['failed'])
    success_count = len(results['success'])
    total_count = failed_count + success_count
    
    print(f"\n📁 {file_name}")
    print(f"   Total variables: {total_count}")
    print(f"   ✅ Successful: {success_count} ({success_count/total_count*100:.1f}%)")
    print(f"   ❌ Failed: {failed_count} ({failed_count/total_count*100:.1f}%)")
    
    if failed_count > 0:
        print(f"   Failed variables:")
        for var in results['failed']:
            print(f"     - {var['var_name']}")

print(f"\n🎯 KEY FINDINGS:")
main_file_results = all_results["./output/conversion_rules.json"]
print(f"   • Main conversion file has {len(main_file_results['success'])} successful conversions")
print(f"   • Only {len(main_file_results['failed'])} variables failed (99.4% success rate)")
print(f"   • The improved rules file has 100% success rate for processed variables")
print(f"   • Overall, the LLM-based conversion is highly successful")

📋 CONVERSION RULES ANALYSIS SUMMARY

📁 conversion_rules.json
   Total variables: 523
   ✅ Successful: 520 (99.4%)
   ❌ Failed: 3 (0.6%)
   Failed variables:
     - VISITDAY
     - OTRLBLI
     - REYTCOR

📁 improved_conversion_rules.json
   Total variables: 10
   ✅ Successful: 10 (100.0%)
   ❌ Failed: 0 (0.0%)

🎯 KEY FINDINGS:
   • Main conversion file has 520 successful conversions
   • Only 3 variables failed (99.4% success rate)
   • The improved rules file has 100% success rate for processed variables
   • Overall, the LLM-based conversion is highly successful


In [7]:
import sys
sys.path.append('./Feature2Txt')

print("🔍 Checking file paths...")
possible_paths = [
    ("./Feature2Txt/data/variable_descriptions_updated.csv", "./Feature2Txt/data/selected_samples_cleaned.csv"),
    ("./data/variable_descriptions_updated.csv", "./data/selected_samples_cleaned.csv"),
    ("../data/variable_descriptions_updated.csv", "../data/selected_samples_cleaned.csv")
]

var_desc_path = None
sample_data_path = None

for var_path, sample_path in possible_paths:
    if os.path.exists(var_path) and os.path.exists(sample_path):
        var_desc_path = var_path
        sample_data_path = sample_path
        print(f"✅ Found files at: {var_path} and {sample_path}")
        break

if not var_desc_path:
    print("❌ Could not find required data files. Searching in current directory...")
    for root, dirs, files in os.walk("."):
        for file in files:
            if "variable_descriptions" in file:
                print(f"  Found variable descriptions: {os.path.join(root, file)}")
            if "selected_samples" in file:
                print(f"  Found sample data: {os.path.join(root, file)}")
    
    print("Please check the file paths and update accordingly.")
else:
    try:
        from generate_rules import LLMClient, process_single_variable, extract_json_from_response
        from config import DEEPSEEK_API_KEY, DEEPSEEK_BASE_URL
        
        print("✅ Successfully imported required functions")
        
        print("\n🔄 REGENERATING FAILED VARIABLE RULES")
        print("=" * 50)

        var_df = pd.read_csv(var_desc_path)
        sample_data = pd.read_csv(sample_data_path)

        variable_descriptions = {}
        for _, row in var_df.iterrows():
            variable_descriptions[row['variable']] = {
                'variable_type': row['variable_type'],
                'short_descriptor': row['short_descriptor'],
                'data_type': row['data_type'],
                'allowable_codes': str(row['allowable_codes']) if pd.notna(row['allowable_codes']) else "",
                'description_or_derivation': str(row['description_or_derivation']) if pd.notna(row['description_or_derivation']) else ""
            }

        failed_vars = all_results["./output/conversion_rules.json"]['failed']
        print(f"Found {len(failed_vars)} failed variables to regenerate:")
        for var in failed_vars:
            print(f"  - {var['var_name']}")

        print(f"\n🔧 Starting regeneration process...")

        regenerated_rules = {}

        for i, failed_var in enumerate(failed_vars, 1):
            var_name = failed_var['var_name']
            print(f"\n[{i}/{len(failed_vars)}] Regenerating: {var_name}")
            
            if var_name not in variable_descriptions:
                print(f"  ⚠️  Warning: No description found for {var_name}")
                continue
            
            var_desc = variable_descriptions[var_name]
            
            if var_name in sample_data.columns:
                unique_values = sample_data[var_name].dropna().unique()
                sample_values = list(unique_values)[:10]
            else:
                sample_values = []
                print(f"  ⚠️  Warning: No sample data found for {var_name}")
            
            print(f"  📊 Sample values: {sample_values}")
            
            args = (var_name, var_desc, sample_values, DEEPSEEK_API_KEY, DEEPSEEK_BASE_URL)
            
            try:
                result_var_name, rule = process_single_variable(args)
                if rule:
                    regenerated_rules[var_name] = rule
                    print(f"  ✅ Successfully regenerated rule for {var_name}")
                    print(f"      English Name: {rule.get('english_name', '')}")
                    print(f"      Template: {rule.get('template', '')}")
                    print(f"      Mappings: {len(rule.get('value_mappings', {}))}")
                else:
                    print(f"  ❌ Failed to regenerate rule for {var_name}")
            except Exception as e:
                print(f"  ❌ Error regenerating {var_name}: {e}")

        print(f"\n📋 REGENERATION SUMMARY:")
        print(f"   Total failed variables: {len(failed_vars)}")
        print(f"   Successfully regenerated: {len(regenerated_rules)}")
        if len(failed_vars) > 0:
            print(f"   Regeneration success rate: {len(regenerated_rules)/len(failed_vars)*100:.1f}%")
        
    except ImportError as e:
        print(f"❌ Import error: {e}")
        print("Please ensure that generate_rules.py and config.py are available in the Feature2Txt directory.")

🔍 Checking file paths...
✅ Found files at: ./data/variable_descriptions_updated.csv and ./data/selected_samples_cleaned.csv
✅ Successfully imported required functions

🔄 REGENERATING FAILED VARIABLE RULES
Found 3 failed variables to regenerate:
  - VISITDAY
  - OTRLBLI
  - REYTCOR

🔧 Starting regeneration process...

[1/3] Regenerating: VISITDAY
  📊 Sample values: [np.float64(23.0), np.float64(7.0), np.float64(11.0), np.float64(17.0), np.float64(5.0), np.float64(6.0), np.float64(27.0), np.float64(12.0), np.float64(25.0), np.float64(30.0)]

[VISITDAY] Conversion Result:
  English Name: Visit Day
  Template: The patient's initial assessment was completed on the {VALUE} of the month.
  Value Mappings: 32 mappings
    1: 1st
    2: 2nd
    3: 3rd
    ... and 29 more
  ✅ Successfully regenerated rule for VISITDAY
      English Name: Visit Day
      Template: The patient's initial assessment was completed on the {VALUE} of the month.
      Mappings: 32

[2/3] Regenerating: OTRLBLI
  📊 Sample

2025-08-07 10:40:42,427 - ERROR - LLM API call failed: HTTPSConnectionPool(host='api.deepseek.com', port=443): Read timed out.


[OTRLBLI] Failed to generate rule: Expecting value: line 1 column 1 (char 0)
[OTRLBLI] Using basic rule as fallback
  ✅ Successfully regenerated rule for OTRLBLI
      English Name: Otrlbli
      Template: The patient has {VALUE}
      Mappings: 1

[3/3] Regenerating: REYTCOR
  📊 Sample values: []


2025-08-07 10:41:12,599 - ERROR - LLM API call failed: HTTPSConnectionPool(host='api.deepseek.com', port=443): Read timed out.


[REYTCOR] Failed to generate rule: Expecting value: line 1 column 1 (char 0)
[REYTCOR] Using basic rule as fallback
  ✅ Successfully regenerated rule for REYTCOR
      English Name: Reytcor
      Template: The patient has {VALUE}
      Mappings: 1

📋 REGENERATION SUMMARY:
   Total failed variables: 3
   Successfully regenerated: 3
   Regeneration success rate: 100.0%


In [10]:
print("\n" + "="*70)
print("📋 DETAILED VIEW OF REGENERATED RULES")
print("="*70)

for var_name, rule in regenerated_rules.items():
    print(f"\n🔹 Variable: {var_name}")
    print(f"   English Name: {rule.get('english_name', 'N/A')}")
    print(f"   Template: {rule.get('template', 'N/A')}")
    print(f"   Description: {rule.get('description', 'N/A')}")
    
    value_mappings = rule.get('value_mappings', {})
    print(f"   Value Mappings ({len(value_mappings)} total):")
    
    for key, value in value_mappings.items():
        if len(value) > 60:
            display_value = value[:60] + "..."
        else:
            display_value = value
        print(f"     '{key}': '{display_value}'")

if regenerated_rules:
    print(f"\n💾 UPDATING CONVERSION RULES FILE")
    print("-" * 40)
    
    rules_file = "./output/conversion_rules.json"
    with open(rules_file, 'r', encoding='utf-8') as f:
        original_rules = json.load(f)
    
    updated_count = 0
    for var_name, new_rule in regenerated_rules.items():
        if var_name in original_rules:
            original_rules[var_name] = new_rule
            updated_count += 1
            print(f"   ✅ Updated rule for: {var_name}")
    
    backup_file = "./output/conversion_rules_backup.json"
    updated_file = "./output/conversion_rules_updated.json"
    
    with open(rules_file, 'r', encoding='utf-8') as f:
        original_backup = json.load(f)
    with open(backup_file, 'w', encoding='utf-8') as f:
        json.dump(original_backup, f, ensure_ascii=False, indent=2)
    print(f"   📁 Created backup: {backup_file}")
    
    with open(updated_file, 'w', encoding='utf-8') as f:
        json.dump(original_rules, f, ensure_ascii=False, indent=2)
    print(f"   💾 Saved updated rules: {updated_file}")
    
    print(f"\n🎯 UPDATE SUMMARY:")
    print(f"   Variables updated: {updated_count}")
    print(f"   Backup created: {backup_file}")
    print(f"   Updated file: {updated_file}")
else:
    print(f"\n⚠️  No rules were successfully regenerated, skipping file update.")


📋 DETAILED VIEW OF REGENERATED RULES

🔹 Variable: VISITDAY
   English Name: Visit Day
   Template: The patient's initial assessment was completed on the {VALUE} of the month.
   Description: The day of the month when Form A1 was completed for the patient's visit.
   Value Mappings (32 total):
     '1': '1st'
     '2': '2nd'
     '3': '3rd'
     '4': '4th'
     '5': '5th'
     '6': '6th'
     '7': '7th'
     '8': '8th'
     '9': '9th'
     '10': '10th'
     '11': '11th'
     '12': '12th'
     '13': '13th'
     '14': '14th'
     '15': '15th'
     '16': '16th'
     '17': '17th'
     '18': '18th'
     '19': '19th'
     '20': '20th'
     '21': '21st'
     '22': '22nd'
     '23': '23rd'
     '24': '24th'
     '25': '25th'
     '26': '26th'
     '27': '27th'
     '28': '28th'
     '29': '29th'
     '30': '30th'
     '31': '31st'
     'default': 'an unspecified day'

🔹 Variable: OTRLBLI
   English Name: Otrlbli
   Template: The patient has {VALUE}
   Description: Information about otrlbli
   

In [11]:
if regenerated_rules:
    print("\n" + "="*60)
    print("🔍 VERIFICATION OF UPDATED RULES")
    print("="*60)
    
    updated_file = "./output/conversion_rules_updated.json" 
    if os.path.exists(updated_file):
        print(f"Analyzing updated file: {updated_file}")
        
        with open(updated_file, 'r', encoding='utf-8') as f:
            updated_rules = json.load(f)
        
        updated_failed = []
        for var_name in [v['var_name'] for v in failed_vars]:
            if var_name in updated_rules:
                rule = updated_rules[var_name]
                value_mappings = rule.get("value_mappings", {})
                english_name = rule.get("english_name", "")
                template = rule.get("template", "")
                
                is_still_failed = False
                if len(value_mappings) == 1 and "default" in value_mappings:
                    default_value = value_mappings["default"].lower()
                    if "unknown" in default_value:
                        is_still_failed = True
                
                expected_simple_name = var_name.replace('_', ' ').title()
                if english_name == expected_simple_name and template == "The patient has {VALUE}":
                    is_still_failed = True
                
                if is_still_failed:
                    updated_failed.append(var_name)
        
        print(f"\n📊 VERIFICATION RESULTS:")
        print(f"   Original failed variables: {len(failed_vars)}")
        print(f"   Successfully regenerated: {len(regenerated_rules)}")
        print(f"   Still failed after update: {len(updated_failed)}")
        
        if updated_failed:
            print(f"   Variables still failing:")
            for var in updated_failed:
                print(f"     - {var}")
        else:
            print(f"   🎉 All previously failed variables have been successfully fixed!")
        
        total_vars = len(updated_rules)
        final_failed_count = len(updated_failed)
        final_success_count = total_vars - final_failed_count
        final_success_rate = (final_success_count / total_vars) * 100
        
        print(f"\n🏆 FINAL STATISTICS:")
        print(f"   Total variables: {total_vars}")
        print(f"   Successful conversions: {final_success_count}")
        print(f"   Failed conversions: {final_failed_count}")
        print(f"   Final success rate: {final_success_rate:.1f}%")
        
        if final_success_rate == 100.0:
            print(f"   🎊 Perfect! All variables now have proper conversion rules!")
        elif final_success_rate >= 99.0:
            print(f"   🌟 Excellent! Nearly all variables have proper conversion rules!")
        else:
            print(f"   👍 Good progress! Success rate improved significantly!")

print(f"\n✨ Task completed! The failed variables have been regenerated using the LLM.")


🔍 VERIFICATION OF UPDATED RULES
Analyzing updated file: ./output/conversion_rules_updated.json

📊 VERIFICATION RESULTS:
   Original failed variables: 3
   Successfully regenerated: 3
   Still failed after update: 2
   Variables still failing:
     - OTRLBLI
     - REYTCOR

🏆 FINAL STATISTICS:
   Total variables: 523
   Successful conversions: 521
   Failed conversions: 2
   Final success rate: 99.6%
   🌟 Excellent! Nearly all variables have proper conversion rules!

✨ Task completed! The failed variables have been regenerated using the LLM.
